# 03 — ConLID: zero-shot, target-only, rehearsal
Loads **EPFL ConLID** pretrained tensors, vocabulary, and label mapping. The encoder remains trainable. Adaptation uses cross-entropy, not a reimplementation of contrastive pretraining.

Run notebook 00 first. Keep the same data and settings for every model. Checkpoints are large; run one model at a time.

Table 2 measures whether forgetting occurs. Table 3 measures mitigation. Neither outcome is assumed.

In [1]:
from pathlib import Path
import os, sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/lid_finetuning_bundle')]
ROOT = next((p for p in candidates if (p/'config.json').exists() and (p/'lidlab').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the complete ZIP first and set ROOT to its lid_finetuning_bundle folder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project folder:', ROOT)


Project folder: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method


In [2]:
from lidlab.data import load_config
from lidlab.experiment import run_experiment
c = load_config()
MODEL = 'conlid'
print('Starting checkpoint:', c['models'][MODEL])
print('Replay starts from:', c['replay_start'])

Starting checkpoint: {'repo_id': 'epfl-nlp/ConLID', 'revision': None, 'local_path': None, 'lr': 0.05, 'existing_zero_predictions': None}
Replay starts from: base


## Run all three conditions
The first run downloads and locks the original checkpoint. Both trained conditions keep every original label and add missing target labels. An extra untrained `initialized` evaluation measures the effect of adding labels alone.

Existing zero-shot predictions are imported only if configured and aligned with the evaluation index. Otherwise zero-shot is recalculated. Completed unchanged stages are reused. Interrupted stages restart from their defined initial checkpoint.

If you change data or hyperparameters, use a new `output_dir` before rerunning.

In [3]:
results = run_experiment(c, MODEL)
for phase, benchmark_results in results.items():
    for benchmark, frame in benchmark_results.items():
        print(phase, benchmark)
        display(frame[['language','precision','recall','f1','support']])

wili-2018.jsonl: NOTE ['arb_Arab'] absent by declaration; scored over 10 categories


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

conlid zero_shot {'commonlid': 0.7104, 'flores_plus': 0.7422, 'wili_2018': 0.7375}
conlid initialized {'commonlid': 0.7104, 'flores_plus': 0.7422, 'wili_2018': 0.7375}


Tokenizing ConLID:   0%|          | 0/60285 [00:00<?, ?it/s]

...

,language,precision,recall,f1,support
0,sin_Sinh,0.957306,0.965837,0.961553,2693
1,pli_Sinh,0.971014,0.973902,0.972456,3027
2,san_Sinh,0.979608,0.900975,0.938648,1333
3,san_Deva,0.955506,0.966254,0.960850,889
4,eng_Latn,0.992754,0.803622,0.888231,27447
5,tam_Taml,0.964286,1.000000,0.981818,81
6,hin_Deva,0.996102,0.906412,0.949143,3665
7,ben_Beng,1.000000,0.929973,0.963716,1885
8,arb_Arab,0.999333,0.977745,0.988421,26061
9,fra_Latn,0.985401,0.835396,0.904220,3232


replay flores_plus


,language,precision,recall,f1,support
0,sin_Sinh,0.957306,0.965837,0.961553,2693
1,pli_Sinh,0.971334,0.973902,0.972616,3027
2,san_Sinh,0.979608,0.905049,0.940854,1327
3,san_Deva,1.000000,0.997036,0.998516,1012
4,eng_Latn,1.000000,0.998024,0.999011,1012
5,tam_Taml,0.999013,1.000000,0.999506,1012
6,hin_Deva,0.997036,0.997036,0.997036,1012
7,ben_Beng,1.000000,1.000000,1.000000,1012
8,arb_Arab,0.997009,0.978474,0.987654,1022
9,fra_Latn,1.000000,1.000000,1.000000,1012


replay wili_2018


,language,precision,recall,f1,support
0,sin_Sinh,0.957306,0.965837,0.961553,2693
1,pli_Sinh,0.971334,0.973902,0.972616,3027
2,san_Sinh,0.979608,0.896269,0.936087,1340
3,san_Deva,0.998983,0.998983,0.998983,983
4,eng_Latn,0.888988,0.993000,0.938120,1000
5,tam_Taml,0.998973,0.989827,0.994379,983
6,hin_Deva,1.000000,0.980981,0.990399,999
7,ben_Beng,1.000000,0.893000,0.943476,1000
8,arb_Arab,0.000000,0.000000,NaN,0
9,fra_Latn,0.985844,0.982863,0.984351,992


## Inspect forgetting and recovery

In [4]:
import pandas as pd
for benchmark in results['zero_shot']:
    base = results['zero_shot'][benchmark].set_index('language')
    target = results['target_only'][benchmark].set_index('language')
    replay = results['replay'][benchmark].set_index('language')
    comparison = pd.DataFrame({'zero_shot_f1':base.f1, 'target_only_f1':target.f1,
                               'replay_f1':replay.f1, 'drop_after_target_only':base.f1-target.f1,
                               'improvement_with_replay':replay.f1-target.f1})
    print(benchmark)
    display(comparison)
print('Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.')

commonlid


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.550937,0.965568,0.961553,-0.414631,-0.004016
pli_Sinh,0.000000,0.974554,0.972456,-0.974554,-0.002098
san_Sinh,0.000000,0.949651,0.938648,-0.949651,-0.011004
san_Deva,0.962372,0.962372,0.960850,0.000000,-0.001522
eng_Latn,0.850219,0.850616,0.888231,-0.000397,0.037615
tam_Taml,0.981818,0.981818,0.981818,0.000000,0.000000
hin_Deva,0.944699,0.944699,0.949143,0.000000,0.004444
ben_Beng,0.962004,0.962004,0.963716,0.000000,0.001712
arb_Arab,0.768311,0.762933,0.988421,0.005378,0.225488


flores_plus


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.550937,0.965568,0.961553,-0.414631,-0.004016
pli_Sinh,0.000000,0.975360,0.972616,-0.975360,-0.002743
san_Sinh,0.000000,0.951863,0.940854,-0.951863,-0.011009
san_Deva,0.998516,0.998516,0.998516,0.000000,0.000000
eng_Latn,0.996032,0.996032,0.999011,0.000000,0.002979
tam_Taml,0.999506,0.999506,0.999506,0.000000,0.000000
hin_Deva,0.997036,0.997036,0.997036,0.000000,0.000000
ben_Beng,1.000000,1.000000,1.000000,0.000000,0.000000
arb_Arab,0.625503,0.618084,0.987654,0.007420,0.369571


wili_2018


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.550937,0.965568,0.961553,-0.414631,-0.004016
pli_Sinh,0.000000,0.975360,0.972616,-0.975360,-0.002743
san_Sinh,0.000000,0.947084,0.936087,-0.947084,-0.010997
san_Deva,0.998473,0.998473,0.998983,0.000000,0.000509
eng_Latn,0.944767,0.944310,0.938120,0.000458,-0.006190
tam_Taml,0.994379,0.994379,0.994379,0.000000,0.000000
hin_Deva,0.990399,0.990399,0.990399,0.000000,0.000000
ben_Beng,0.943476,0.943476,0.943476,0.000000,0.000000
arb_Arab,NaN,NaN,NaN,NaN,NaN


Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.


After all three model notebooks finish, run notebook 04. Do not infer preservation of all original languages from eight replay-language scores. The original label set remains available, but retention needs evaluation.